In [1]:
!pip install sentence_transformers
%pip install pyarrow
%pip install --use-pep517 annoy
%pip install tensorflow
%pip install pydot
%pip install pydot
%pip install tflite_runtime
%pip install tokenizers


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[no

In [2]:
%pip install pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
import tensorflow as tf  # Use normal TensorFlow
from tokenizers import Tokenizer

class TFLiteTextEmbedder:
    def __init__(self, model_path: str, tokenizer_path: str, max_length: int = 128):
        # Load tokenizer with padding
        self.tokenizer = Tokenizer.from_file(tokenizer_path)
        self.tokenizer.enable_padding(pad_id=0, pad_token="[PAD]", length=max_length)
        self.max_length = max_length
        
        # Load TFLite model (will allocate later)
        self.interpreter = tf.lite.Interpreter(model_path=model_path)
        self.input_details = self.interpreter.get_input_details()
        self.output_details = self.interpreter.get_output_details()
        
        # Model will be allocated on first run
        self.allocated = False

    def encode(self, queries):
        if isinstance(queries, str):
            queries = [queries]
    
        # Tokenize
        encoded_input = self.tokenizer.encode_batch(queries)
        input_ids = np.array([e.ids for e in encoded_input], dtype=np.int32)
        attention_mask = np.array([e.attention_mask for e in encoded_input], dtype=np.int32)
    
        # Ensure batch dimension
        if input_ids.ndim == 1:
            input_ids = np.expand_dims(input_ids, axis=0)
        if attention_mask.ndim == 1:
            attention_mask = np.expand_dims(attention_mask, axis=0)
    
        # Always resize & allocate tensors to match current batch
        self.interpreter.resize_tensor_input(self.input_details[0]['index'], input_ids.shape)
        if len(self.input_details) > 1:
            self.interpreter.resize_tensor_input(self.input_details[1]['index'], attention_mask.shape)
        self.interpreter.allocate_tensors()
    
        self.interpreter.set_tensor(self.input_details[0]['index'], input_ids)
        if len(self.input_details) > 1:
            self.interpreter.set_tensor(self.input_details[1]['index'], attention_mask)
    
        self.interpreter.invoke()
        embeddings = self.interpreter.get_tensor(self.output_details[0]['index'])
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        return embeddings / norms

# Example usage
model = TFLiteTextEmbedder(
    model_path="./models/all-MiniLM-L6-v2-quant.tflite",
    tokenizer_path="./models/tokenizers/all-MiniLM-L6-v2_tokenizer.json",
    max_length=128
)

texts = "Hello world"
embeddings = model.encode(texts)
print(embeddings)

[[-3.73296142e-02  3.57798152e-02  2.63164490e-02  1.24688083e-02
  -2.06889361e-02 -1.54800937e-01  7.46317208e-02 -6.48693135e-03
  -4.28984687e-02  2.46554613e-02  5.62800281e-02 -1.29850709e-03
   3.88236046e-02  9.95423459e-03  1.56517513e-02 -3.11795324e-02
   1.50670512e-02 -2.47750282e-02 -1.69485047e-01 -1.11889224e-02
   1.35889684e-03  6.46715537e-02  2.93410500e-03  2.38831006e-02
  -7.90916085e-02  2.39872257e-03  5.11206686e-02  4.95913811e-02
  -2.76270811e-03 -3.92039865e-02  3.49156708e-02  1.63786355e-02
   1.29850432e-01 -4.05161455e-02  2.47526057e-02  3.21611799e-02
  -2.20841896e-02 -1.02325864e-01 -3.98896746e-02  3.11185829e-02
   5.47225364e-02 -6.79136962e-02 -2.31208969e-02 -2.06276476e-02
  -1.38958485e-03 -1.83591675e-02 -5.61563717e-03  2.26931777e-02
   6.20412827e-02 -8.43195710e-03 -4.65284400e-02 -6.37134090e-02
   2.04106439e-02 -8.34550615e-03  5.80396391e-02  1.54289575e-02
   6.16759667e-03 -5.92275076e-02  1.32408720e-02 -2.73474064e-02
  -3.35915

/usr/local/lib/python3.11/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [2]:
import os
import gc
from annoy import AnnoyIndex
import pandas as pd
import glob
import numpy as np
from sentence_transformers import SentenceTransformer, util
import pandas as pd

for root, dirs, files in os.walk("."):
    for name in files:
        print(os.path.join(root, name))

./query_7.csv
./product_113.csv
./product_107.csv
./product_82.csv
./product_96.csv
./product_41.csv
./product_55.csv
./product_69.csv
./product_68.csv
./product_54.csv
./product_40.csv
./product_97.csv
./product_83.csv
./product_106.csv
./product_112.csv
./query_6.csv
./query_4.csv
./product_104.csv
./product_110.csv
./product_138.csv
./product_95.csv
./product_81.csv
./product_56.csv
./product_42.csv
./product_43.csv
./product_57.csv
./product_80.csv
./product_94.csv
./product_139.csv
./product_111.csv
./product_105.csv
./Cleaning.ipynb
./query_5.csv
./query_1.csv
./product_129.csv
./product_101.csv
./product_115.csv
./product_90.csv
./product_84.csv
./product_53.csv
./product_47.csv
./product_46.csv
./product_52.csv
./product_85.csv
./product_91.csv
./query_model.tflite
./product_114.csv
./product_100.csv
./product_128.csv
./query_2.csv
./product_116.csv
./product_102.csv
./product_87.csv
./product_93.csv
./product_78.csv
./product_44.csv
./product_50.csv
./product_51.csv
./product_

In [5]:
df_queries_table = pd.read_parquet('../shopping_queries_dataset/shopping_queries_dataset_examples.parquet')
df_reduced_queries_table = df_queries_table[df_queries_table["small_version"] == 1]

df_products_table = pd.read_parquet('../shopping_queries_dataset/shopping_queries_dataset_products.parquet')
df_metaData_table = pd.read_csv("../shopping_queries_dataset/shopping_queries_dataset_sources.csv")

df_reduced_queries_table = df_reduced_queries_table[df_queries_table["product_locale"] == "us"]
df_reduced_queries_table = df_reduced_queries_table[(df_reduced_queries_table.esci_label == 'E') | (df_reduced_queries_table.esci_label == 'I')]

/tmp/ipykernel_1944/4058875514.py:7: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_reduced_queries_table = df_reduced_queries_table[df_queries_table["product_locale"] == "us"]


In [6]:
df_joined = pd.merge(
    df_reduced_queries_table.head(),
    df_products_table,
    how="left",
    on=["product_id", "product_id"]
)

In [7]:
df_joined.head()

,example_id,query,query_id,product_id,product_locale_x,esci_label,small_version,large_version,split,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale_y
0,16,!awnmower tires without rims,1,B075SCHMPY,us,I,1,1,train,"RamPro 10"" All Purpose Utility Air Tires/Wheel...","<b>About The Ram-Pro All Purpose Utility 10"" A...",✓ The Ram-Pro Ten Inch ready to install Air Ti...,RamPro,10 Inch,us
1,17,!awnmower tires without rims,1,B08L3B9B9P,us,E,1,1,train,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,MaxAuto 2-Pack 13x5.00-6 2PLY Turf Mower Tract...,Please check your existing tire Sidewall for t...,MaxAuto,NaN,us
2,18,!awnmower tires without rims,1,B082K7V2GZ,us,I,1,1,train,NEIKO 20601A 14.5 inch Steel Tire Spoon Lever ...,NaN,[QUALITY]: Hardened Steel-Iron construction wi...,Neiko,NaN,us
3,20,!awnmower tires without rims,1,B07C1WZG12,us,E,1,1,train,(Set of 2) 15x6.00-6 Husqvarna/Poulan Tire Whe...,No fuss. Just take off your old assembly and r...,Tire size:15x6.00-6 Ply: 4 Tubeless\n6x4.5 Whe...,Antego Tire & Wheel,Husqvarna Silver,us
4,21,!awnmower tires without rims,1,B077QMNXTS,us,E,1,1,train,MaxAuto 2 Pcs 16x6.50-8 Lawn Mower Tire for Ga...,<br>Tire Specifications:<br> 1. Material: Rubb...,"Set of 2 16X6.50-8, 16x6.50x8, 16-6.50-8 Lawn ...",MaxAuto,Black,us


In [8]:
def find_embeddings(lst_product_title,i, maxlen):
    # Define a list of sentences
    sentences = list(lst_product_title)[i:i+step]
    sentence_embeddings = model.encode(sentences)

    return np.array(sentence_embeddings)

In [9]:
# Load a pre-trained model (you can choose from various models like BERT, RoBERTa, etc.)


In [10]:
# Create sentence embeddings for your sentences
sentences = ["This is an example sentence.", "Handling unknown words in embeddings is important."]
embeddings = model.encode(sentences)

# The 'embeddings' variable now contains the sentence embeddings as PyTorch tensors
print(embeddings)
print(embeddings.shape)


[[ 7.97536299e-02  4.03765962e-02  5.12124747e-02  9.40882340e-02
   4.19790000e-02 -1.52013758e-02  1.36489468e-02 -8.01432598e-03
   7.64647275e-02  6.27813803e-04  5.35987616e-02 -2.79270094e-02
   3.25311720e-02  5.26431995e-03  5.44801168e-02  1.46142915e-02
   4.73702028e-02 -1.10584050e-02 -8.52930322e-02  2.36989520e-02
   2.89930310e-03  1.87786426e-02  2.42492296e-02  1.57118123e-02
  -2.46948078e-02 -3.74782761e-03 -4.61851545e-02  6.08019158e-02
   7.81291276e-02 -3.89705375e-02 -7.20776245e-02 -4.72738408e-02
   5.84090315e-02  5.18623292e-02  8.49146047e-04  3.63453627e-02
  -5.60104800e-03  8.40311944e-02 -3.34232226e-02  2.20274343e-03
   1.38214035e-02  1.41607458e-02  1.22072529e-02 -2.20712628e-02
   2.93678232e-02 -5.77052496e-02  9.80309676e-03  1.06284115e-02
   5.65856099e-02 -3.90197709e-02 -8.65615606e-02 -5.43642715e-02
  -8.76574963e-02  2.09595938e-03  1.00091612e-02  7.45996907e-02
   3.16641778e-02  6.86959252e-02  3.19658481e-02 -1.98435821e-02
  -2.62036

In [11]:
sample_size = 150000
df_queries_dataset_mini = df_reduced_queries_table.sample(sample_size, random_state = 42).reset_index(drop=True)
df_queries_dataset_mini.shape

(150000, 9)

In [12]:
product_cols = ['product_title', 'product_description', 'product_id']
df_products_dataset_mini = pd.merge(df_products_table[product_cols].drop_duplicates(), df_queries_dataset_mini[['product_id']].drop_duplicates() ,on = ['product_id'])
df_products_dataset_mini.shape

(139639, 3)

In [13]:
null_counts = df_products_dataset_mini.isnull().sum()
print(null_counts)

df_products_dataset_mini['product_title'] = df_products_dataset_mini['product_title'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_products_dataset_mini['product_description'] = df_products_dataset_mini['product_description'].apply(lambda x : str(x).lower() if pd.notna(x) else '')
df_queries_dataset_mini['query'] = df_queries_dataset_mini['query'].apply(lambda x : str(x).lower() if pd.notna(x) else '')

null_counts = df_products_dataset_mini.isnull().sum()
print(null_counts)

product_title              0
product_description    68703
product_id                 0
dtype: int64
product_title          0
product_description    0
product_id             0
dtype: int64


In [ ]:
queries = list(df_queries_dataset_mini['query'].unique())

step = 1000
query_dim = 384
1
cols = ['q' + str(x) for x in list(range(0, query_dim))] + ['query']
cnt = 0

for i in range(0,len(queries),step):
    
    cnt += 1
    # Define a list of sentences
    sentences = list(queries)[i:i+step]

    sentence_embeddings = model.encode(sentences)
    
    df_tmp = pd.DataFrame(np.concatenate((sentence_embeddings, np.array(sentences).reshape(-1,1)), axis=1))
    
    df_tmp.columns = cols
    
    df_tmp.to_csv(
        f'query_{cnt}.csv', header = True, index = False)
    
#     if cnt == 2:
#         break
        
    print(i)

In [14]:
gc.collect()

80

In [15]:
lst_product_title = list(df_products_dataset_mini['product_title'])
lst_product_description = list(df_products_dataset_mini['product_description'])
lst_product_id = list(df_products_dataset_mini['product_id'])
df_products_dataset_mini.head()

,product_title,product_description,product_id
0,spa-cinco pies de ti / 5 feet,,1644730162
1,fushing 30 transparentes piezas plásticas de v...,<b>los titulares de la etiqueta de nombre con ...,B01M0JC3OQ
2,marina - kit de acuario con iluminación led 20...,,B0173I55Q0
3,"aftershokz aeropex, auriculares deportivos ina...",,B07RQLRV7Q
4,pinowu - lote de 2 cordones de sujeción magnét...,<b>pinowu - cordones de sujeción para cortina ...,B07RTS9TLP


In [16]:
print(len(lst_product_title))

139639


In [ ]:
step = 1000
product_dim = 384

cols = ['p' + str(x) for x in list(range(0, product_dim * 2))] + ['product_id']\

cnt = 0
for i in range(0,len(lst_product_title),step):
    cnt += 1
    product_title_embed = find_embeddings(lst_product_title, i, "s")
    product_description_embed = find_embeddings(lst_product_description, i, "")
    
    # Concatenate arrays column-wise and reshape lst_product_id to (x, 1)
    df_tmp = pd.DataFrame(np.concatenate((
        product_title_embed,
        product_description_embed,
        np.array(lst_product_id[i:i + step]).reshape(-1, 1)
    ), axis=1))
    
    
    df_tmp.columns = cols
    df_tmp.to_csv(f'product_{cnt}.csv', header = True, index = False)
    
#     if (cnt == 2):
#         break
    
    print(i)

In [17]:
gc.collect()

0

In [ ]:
# Get a list of CSV files that start with "product_"
file_list = glob.glob('product_*.csv')

# Initialize an empty list to store DataFrames
dfs = []

# Read each CSV file and append it to the list
for file in file_list:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all DataFrames into one
concatenated_product_df = pd.concat(dfs, ignore_index=True)

# Save the concatenated DataFrame to a new CSV file
concatenated_product_df.to_csv('product_embeddings.csv', index=False)

print(concatenated_product_df.shape)

concatenated_product_df.head()

In [ ]:
# Get a list of CSV files that start with "product_"
file_list = glob.glob('query_*.csv')

# Initialize an empty list to store DataFrames
dfs = []

# Read each CSV file and append it to the list
for file in file_list:
    df = pd.read_csv(file)
    dfs.append(df)

# Concatenate all DataFrames into one
concatenated_query_df = pd.concat(dfs, ignore_index=True)

# Save the concatenated DataFrame to a new CSV file
concatenated_query_df.to_csv('query_embeddings.csv', index=False)

print(concatenated_query_df.shape)

concatenated_query_df.head()

In [19]:
# Load query embeddings
query_df = pd.read_csv('./query_embeddings.csv')

# Ensure no duplicate queries
query_df = query_df.drop_duplicates(subset=['query'])

df_queries_dataset_mini = pd.merge(
    df_queries_dataset_mini,
    query_df,
    on='query',
    how='left'  # safer than inner
)

del query_df


# Load product embeddings
product_df = pd.read_csv('./product_embeddings.csv')

# Ensure no duplicate product IDs
product_df = product_df.drop_duplicates(subset=['product_id'])

df_queries_dataset_mini = pd.merge(
    df_queries_dataset_mini,
    product_df,
    on='product_id',
    how='left'
)

del product_df


# Optional: check for missing embeddings
print("Missing query embeddings:", df_queries_dataset_mini['query'].isna().sum())
print("Missing product embeddings:", df_queries_dataset_mini['product_id'].isna().sum())

# Final shape
print(df_queries_dataset_mini.shape)

# Save
df_queries_dataset_mini.to_csv('dataset_mini.csv', index=False)

/tmp/ipykernel_1768/4038461719.py:18: DtypeWarning: Columns (0: product_title) have mixed types. Specify dtype option on import or set low_memory=False.
  product_df = pd.read_csv('./product_embeddings.csv')


Missing query embeddings: 0
Missing product embeddings: 0
(150000, 1163)


In [18]:
df = pd.read_csv("./dataset_mini.csv")
df.head()
df.shape

(150000, 1163)

In [ ]:
import pandas as pd
import gc

# Load only necessary columns FIRST
df_products_table = pd.read_parquet(
    '../shopping_queries_dataset/shopping_queries_dataset_products.parquet',
    columns=['product_id', 'product_title']
)

# Load embeddings
df_product_embeddings = pd.read_csv('./product_embeddings.csv')

# 🔥 IMPORTANT: drop duplicates BEFORE merge
df_product_embeddings = df_product_embeddings.drop_duplicates()

# Merge (smaller now)
df_product_embedding = pd.merge(
    df_product_embeddings,
    df_products_table,
    on='product_id'
)

# Free memory ASAP
del df_product_embeddings
del df_products_table
gc.collect()

# Add id column (no reset_index needed)
df_product_embedding['pid'] = range(len(df_product_embedding))

/tmp/ipykernel_1944/233025735.py:11: DtypeWarning: Columns (0: product_title) have mixed types. Specify dtype option on import or set low_memory=False.
  df_product_embeddings = pd.read_csv('./product_embeddings.csv')


In [ ]:
df_query_embedding = pd.read_csv('./query_embeddings.csv')

# Drop duplicates early
df_query_embedding = df_query_embedding.drop_duplicates()

# Add id
df_query_embedding['qid'] = range(len(df_query_embedding))

In [ ]:
df_product_embedding.head()

In [ ]:
print(df_query_embedding.shape, df_product_embedding.shape)

In [ ]:
query_tower_input_dim = 384
product_tower_input_dim = (384*2)

q = AnnoyIndex(query_tower_input_dim, 'dot')
mp_query_dict = {}


for ix,row in df_query_embedding.iterrows():
    mp_query_dict[row['qid']] = row['query']
    
    key = int(row['qid'])
    vec = list(row[['q'+str(x) for x in list(range(query_tower_input_dim))]])
    
    #     print(key,vec)
    q.add_item(key,vec)


q.build(100) # 100 trees
q.save('query.tree')

top_k = 20
mat = []
for ix,row in df_query_embedding.iterrows():
    item = row['query']
    mat.append([item] + [mp_query_dict[x] for x in q.get_nns_by_item(row['qid'], top_k+1)[1:]])
    
    if ix == 50:
        break
    
cols = ['query_id']
for i in range(top_k):
    cols += ['nearest_{}'.format(i+1)]

df_neighbors1 = pd.DataFrame(mat, columns = cols)

display(df_neighbors1.head(50))

In [ ]:
p = AnnoyIndex(product_tower_input_dim, 'dot')
mp_product_dict = {}

for ix,row in df_product_embedding.iterrows():
    mp_product_dict[int(row['pid'])] = row['product_title']
    
    key = int(row['pid'])
    vec = list(row[['p'+str(x) for x in list(range(product_tower_input_dim))]])
    
#     print(key,vec)
    p.add_item(key,vec)
    if ix == 50:
        break
    

p.build(100) # 100 trees
p.save('product.tree')


p = AnnoyIndex(product_tower_input_dim,  'euclidean')
p.load('product.tree')



top_k = 20
mat = []
for ix,row in df_product_embedding.iterrows():
    item = row['product_title']
    mat.append([item] + [mp_product_dict[x] for x in p.get_nns_by_item(row['pid'], top_k+1)[1:]])
    
    if ix == 50:
        break
    
cols = ['product_id']
for i in range(top_k):
    cols += ['nearest_{}'.format(i+1)]

print(cols)

df_neighbors2 = pd.DataFrame(mat, columns = cols)

display(df_neighbors2.head(200))

###### 